In [0]:
%sql
USE SCHEMA default;
CREATE VOLUME IF NOT EXISTS landing;

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

DataFrame[]

In [0]:
base_path = "/Volumes/workspace/default/landing"

In [0]:
from pyspark.sql import functions as F

def ingest_to_bronze(file_name, table_name):
    """
    Lê um arquivo CSV da Landing Zone e salva na camada Bronze como Delta.
    overwriteSchema=true garante tolerância a mudanças de esquema entre execuções.
    """
    landing_path = f"/Volumes/workspace/default/landing/{file_name}"
    
    #Leitura dos dados
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(landing_path))
    
    #Adição da coluna de timestamp de ingestão
    df_with_timestamp = df.withColumn("timestamp_ingestion", F.current_timestamp())
    
    #Escrita na camada Bronze no formato Delta
    # overwriteSchema=true permite alterações de esquema sem erro
    (df_with_timestamp.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"bronze.{table_name}"))
    
    print(f"Tabela bronze.{table_name} criada com sucesso.")


In [0]:
# Lista de mapeamento (Arquivo Original, Nome da Tabela)
mapeamento = [
    ("olist_customers_dataset.csv", "tb_customers"),
    ("olist_geolocation_dataset.csv", "tb_geolocalizacao"),
    ("olist_order_items_dataset.csv", "tb_order_items"),
    ("olist_order_payments_dataset.csv", "tb_order_payments"),
    ("olist_order_reviews_dataset.csv", "tb_order_reviews"),
    ("olist_orders_dataset.csv", "tb_orders"),
    ("olist_products_dataset.csv", "tb_products"),
    ("olist_sellers_dataset.csv", "tb_sellers"),
    ("product_category_name_translation.csv", "tb_product_category_name_translation")
]

# Loop para processar todos os arquivos
for arquivo, tabela in mapeamento:
    try:
        ingest_to_bronze(arquivo, tabela)
    except Exception as e:
        print(f"Erro ao processar {arquivo}: {e}")

Tabela bronze.tb_customers criada com sucesso.
Tabela bronze.tb_geolocalizacao criada com sucesso.
Tabela bronze.tb_order_items criada com sucesso.
Tabela bronze.tb_order_payments criada com sucesso.
Tabela bronze.tb_order_reviews criada com sucesso.
Tabela bronze.tb_orders criada com sucesso.
Tabela bronze.tb_products criada com sucesso.
Tabela bronze.tb_sellers criada com sucesso.
Tabela bronze.tb_product_category_name_translation criada com sucesso.


In [0]:
import requests
from pyspark.sql import functions as F

#Criar os parâmetros (widgets) no notebook do Databricks
#O formato exigido pela API MM-DD-AAAA
dbutils.widgets.text("data_inicio", "01-01-2016", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "12-31-2018", "Data Fim (MM-DD-AAAA)")

# 2. Capturar os valores digitados nos parâmetros
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

#Montar a URL da API substituindo as variáveis
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

#Fazer a requisição para a API do Banco Central
print(f"Buscando dados de {data_inicio_formatada} até {data_fim_formatada}...")
response = requests.get(url)

#Verifica se a requisição deu certo (Status 200 = OK)
if response.status_code == 200:
    dados_json = response.json()
    valores = dados_json.get("value", [])
    
    if valores:
        #Converter a lista de dicionários para um DataFrame do Spark
        df_dolar = spark.createDataFrame(valores)
        
        #Adicionar a coluna de timestamp de ingestão (padrão da camada Bronze)
        df_dolar = df_dolar.withColumn("timestamp_ingestion", F.current_timestamp())
        
        # Salvar na camada Bronze no formato Delta
        # overwriteSchema=true garante tolerância a mudanças de esquema
        (df_dolar.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable("bronze.tb_cotacao_dolar"))
         
        print("Tabela bronze.tb_cotacao_dolar criada com sucesso!")
        display(df_dolar)
    else:
        print("A API não retornou cotações para este período. Verifique as datas.")
else:
    print(f"Erro ao acessar a API. Código de status: {response.status_code}")


Buscando dados de 01-01-2026 até 12-31-2026...
Tabela bronze.tb_cotacao_dolar criada com sucesso!


cotacaoCompra,dataHoraCotacao,timestamp_ingestion
5.4366,2026-01-02 13:03:33.254,2026-04-07T19:14:49.197Z
5.4345,2026-01-05 13:10:14.896,2026-04-07T19:14:49.197Z
5.3791,2026-01-06 13:12:07.358,2026-04-07T19:14:49.197Z
5.3874,2026-01-07 13:06:27.001,2026-04-07T19:14:49.197Z
5.3854,2026-01-08 13:04:28.616,2026-04-07T19:14:49.197Z
5.3701,2026-01-09 13:09:29.054,2026-04-07T19:14:49.197Z
5.3754,2026-01-12 13:08:29.056,2026-04-07T19:14:49.197Z
5.3758,2026-01-13 13:08:54.165,2026-04-07T19:14:49.197Z
5.3789,2026-01-14 13:04:32.411,2026-04-07T19:14:49.197Z
5.384,2026-01-15 13:05:25.66,2026-04-07T19:14:49.197Z
